# Phase 10 — Final Held-Out Baseline Evaluation

MorphoNet-BT controlled from-scratch comparison. Five architectures × five patient-separated outer folds. Evaluation only.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.


## Objective and controlled protocol

Evaluate only the validation-selected `best_macro_f1.pt` for each run. No checkpoint reselection, training, tuning, ablation or significance testing is performed. Frozen Phase 7 predictions provide the MorphoNet-BT reference. All results describe this dataset, fixed splits and from-scratch protocol.

Test input is MRI only: one grayscale channel, bilinear resize to 256 × 256, per-image Z-score, no clipping and no augmentation. Masks and morphology targets do not enter inference. Class order is Meningioma, Glioma, Pituitary (0, 1, 2).

## Imports and reproducibility

In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
import os, sys
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
from src.models.baselines import BASELINES, DISPLAY_NAMES
from src.evaluation.baseline_protocol import preflight, evaluate_fold
from src.evaluation.baseline_comparison import analyze, compare, final_audit
pd.set_option('display.max_columns', 15)
pd.set_option('display.precision', 4)
OUT = ROOT / 'results/baselines/evaluation'

## Frozen artifact verification, checkpoint discovery and test integrity

Before any test inference, all 25 best checkpoints must match their completion records, configuration fingerprint, class mapping, preprocessing version and fold. Patient separation and frozen source/split hashes are mandatory. Historical Phase 9 trainer hashes are accepted only through its audited AMP-overflow compatibility rule. No optimizer state is applied. Protected files are hashed before and after evaluation.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
context = preflight(ROOT)
display(context['checkpoints'][['Model','Fold','BestEpoch','ValidationMacroF1','TestPatients','TestSlices']])

## Evaluation utilities and execution policy

Inference uses FP32, batch size 16, deterministic algorithms, disabled TF32, zero workers and no augmentation, matching Phase 7. A completed receipt binds predictions to checkpoint, split and inference source hashes. Rerunning this notebook reuses verified predictions without another test pass. An interrupted, unreceipted pass stops for review. Each of the following cells performs five independent held-out fold evaluations.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
all_frames = {}

## ResNet18 evaluation

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
all_frames['resnet18'] = {fold: evaluate_fold(context, 'resnet18', fold) for fold in range(1, 6)}

## ResNet50 evaluation

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
all_frames['resnet50'] = {fold: evaluate_fold(context, 'resnet50', fold) for fold in range(1, 6)}

## DenseNet121 evaluation

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
all_frames['densenet121'] = {fold: evaluate_fold(context, 'densenet121', fold) for fold in range(1, 6)}

## MobileNetV3-Small evaluation

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
all_frames['mobilenet_v3_small'] = {fold: evaluate_fold(context, 'mobilenet_v3_small', fold) for fold in range(1, 6)}

## EfficientNet-B0 evaluation

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
all_frames['efficientnet_b0'] = {fold: evaluate_fold(context, 'efficientnet_b0', fold) for fold in range(1, 6)}

## Per-model OOF integrity and metric computation

All five models must pass the coverage gate before comparative results are produced: 3,064 unique samples, 233 patients, no missing/unexpected samples and one outer test fold per patient. Pooled metrics are recomputed from concatenated predictions; fold mean and sample SD (ddof=1) remain separate.

Macro ROC-AUC is the unweighted mean of three one-vs-rest AUCs. Specificity is TN/(TN+FP). Patient predictions use the argmax of mean slice softmax probabilities accumulated in float64. All slices remain included.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
oofs, results = analyze(context, all_frames)
display(pd.DataFrame(context['metadata']['oof_integrity_results']).T)

## Frozen reference and comparison preparation

Read exact Phase 7 CSV results and verify them against stored predictions without rerunning MorphoNet-BT. Reuse Phase 5/8 parameter, FP32 parameter-storage and Conv/Linear MAC/FLOP counts; do not profile models again. MACs exclude operations outside convolution/linear layers; storage is parameter-only, not runtime memory or checkpoint size.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
tables = compare(context, oofs, results)

## Primary slice-level comparison

These are pooled OOF metrics, not fold means. The table includes every requested primary metric.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['final_model_comparison'])

## Five-fold mean ± sample SD

FoldMean and FoldSD summarize five outer test folds; sample SD uses ddof=1. AggregatedOOFValue is calculated separately.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['five_fold_mean_sd'])

## Slice-level per-class metrics

Sensitivity is recall. Support counts slices; class order remains fixed.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['per_class_comparison'])

## Patient-level comparison

Secondary analysis: one mean-probability prediction per patient, 233 patients per model.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['patient_level_model_comparison'])

## Patient-level per-class metrics

Patient support counts patients, not slices.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['patient_per_class_comparison'])

## Efficiency and relative parameter reduction

Parameter reduction is (1 − proposed parameters / baseline parameters) × 100. Ratios and percentage reductions have different meanings. MobileNetV3-Small has fewer MACs than MorphoNet-BT.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['efficiency_comparison'])

## Meningioma analysis

Descriptive class-specific performance; no test-driven changes are permitted.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['meningioma_comparison'])

## Fold 2 analysis

Descriptive held-out performance for the same outer fold. No experiments are altered based on these results.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['fold_2_comparison'])

## Misclassifications

All difficult cases are retained. Detailed slice and patient error files are saved for each baseline.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['misclassification_counts'])

## Descriptive error overlap

Both-correct and discordant-error counts describe paired predictions only; they are not significance tests.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['error_overlap'])

## Patient-cluster bootstrap confidence intervals

95% percentile intervals use 2,000 draws, seed 42, resampling 233 patients with replacement and retaining every selected patient’s full slice cluster with multiplicity. Identical sorted-PID draws are shared across all six models and reproduce Phase 7 exactly. Intervals condition on fixed trained models/OOF predictions and exclude training variability.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['accuracy_cluster_bootstrap_ci'])

## Confusion matrices and one-vs-rest ROC analysis

Each baseline has raw and row-normalized slice confusion matrices, a patient confusion matrix, and three-class OVR ROC curves. Rows are true classes and columns predicted classes. Raw and normalized matrices are saved as CSV; figures are 400-dpi PNG and vector PDF. All five models are shown below.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(Image(filename=str(OUT / 'resnet18/figures/oof_confusion_matrix.png'), width=480))
display(Image(filename=str(OUT / 'resnet18/figures/oof_confusion_matrix_normalized.png'), width=480))
display(Image(filename=str(OUT / 'resnet18/figures/patient_confusion_matrix.png'), width=480))
display(Image(filename=str(OUT / 'resnet18/figures/oof_roc_curves.png'), width=480))

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(Image(filename=str(OUT / 'resnet50/figures/oof_confusion_matrix.png'), width=480))
display(Image(filename=str(OUT / 'resnet50/figures/oof_confusion_matrix_normalized.png'), width=480))
display(Image(filename=str(OUT / 'resnet50/figures/patient_confusion_matrix.png'), width=480))
display(Image(filename=str(OUT / 'resnet50/figures/oof_roc_curves.png'), width=480))

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(Image(filename=str(OUT / 'densenet121/figures/oof_confusion_matrix.png'), width=480))
display(Image(filename=str(OUT / 'densenet121/figures/oof_confusion_matrix_normalized.png'), width=480))
display(Image(filename=str(OUT / 'densenet121/figures/patient_confusion_matrix.png'), width=480))
display(Image(filename=str(OUT / 'densenet121/figures/oof_roc_curves.png'), width=480))

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(Image(filename=str(OUT / 'mobilenet_v3_small/figures/oof_confusion_matrix.png'), width=480))
display(Image(filename=str(OUT / 'mobilenet_v3_small/figures/oof_confusion_matrix_normalized.png'), width=480))
display(Image(filename=str(OUT / 'mobilenet_v3_small/figures/patient_confusion_matrix.png'), width=480))
display(Image(filename=str(OUT / 'mobilenet_v3_small/figures/oof_roc_curves.png'), width=480))

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(Image(filename=str(OUT / 'efficientnet_b0/figures/oof_confusion_matrix.png'), width=480))
display(Image(filename=str(OUT / 'efficientnet_b0/figures/oof_confusion_matrix_normalized.png'), width=480))
display(Image(filename=str(OUT / 'efficientnet_b0/figures/patient_confusion_matrix.png'), width=480))
display(Image(filename=str(OUT / 'efficientnet_b0/figures/oof_roc_curves.png'), width=480))

## Paired data for later statistical analysis

`statistical_preparation/aligned_slice_predictions.csv` pairs 3,064 rows by sample ID and PID; the patient counterpart pairs 233 PIDs. Identity, truth and fold are checked exactly. Both contain each model’s predicted class and three probabilities. Shared bootstrap indices, PID order and replicate accuracies are retained. No independent slice tests, McNemar tests or significance claims are made.

## Publication tables and figures

Exact numerical CSV tables and formatted HTML/LaTeX tables are saved. Best metric values are determined programmatically in `tables/metric_leaders.csv`. All charts have separate, explicit units and zero-based axes; parameter counts and accuracy never share an axis.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
for name in ['oof_macro_f1','oof_accuracy','patient_macro_f1','parameters','macs']:
    display(Image(filename=str(OUT / f'figures/{name}.png'), width=700))

## Final scientific audit

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
audit = final_audit(context)
display(pd.Series(audit, name='Verified value'))

## Conclusions and interpretation

The following descriptive leaders are derived from the final tables. Differences in performance do not establish statistical significance. This controlled comparison supports subsequent prespecified ablation and patient-aware paired statistical analysis after separate approval. No such analysis is started here.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
slice_table = tables['final_model_comparison'].set_index('Model')
patient_table = tables['patient_level_model_comparison'].set_index('Model')
print('Highest OOF macro F1:', slice_table['OOF Macro F1'].idxmax())
print('Highest patient macro F1:', patient_table['Patient Macro F1'].idxmax())
print('Smallest parameter count:', tables['efficiency_comparison'].set_index('Model')['Parameters'].idxmin())
print('Lowest Conv/Linear MAC count:', tables['efficiency_comparison'].set_index('Model')['MACs'].idxmin())
print('Final results retained. No retraining, ablation or significance testing performed.')